# 02b. 트랜스포머 모델 비교 실험 (8프레임 @1fps)

클립을 **초당 1프레임 → 8장**으로 뽑아 패치 토큰 기반 비디오 트랜스포머에 태우고,
성능 좋은 사전학습 모델 여러 개를 **같은 조건**에서 비교합니다 (동일 데이터·라벨·평가).

| 후보 | 프레임 | 특징 |
|---|---|---|
| `r2plus1d` | 8 | CNN 기준선 (02 노트북과 비교 anchor) |
| `timesformer` | 8 | 시공간 분리 어텐션, K400 사전학습이 **8프레임×224** — 이 실험과 정합 (Bertasius et al. ICML 2021) |
| `videomae` | 16 | 마스크드 사전학습, 데이터 효율 높음 (Tong et al. NeurIPS 2022) — 16프레임 필요해 2fps 샘플 |
| `clip_frozen` | 8 | CLIP ViT-B/16 **동결** + 2층 시간 트랜스포머 헤드 — 과적합에 가장 안전, 학습 최속 (Radford et al. ICML 2021) |

T4 기준 배치·AMP 조정돼 있음. 결과는 라벨별 AP 표로 저장 → 잘 나오는 걸 채택.

In [ ]:
!pip -q install transformers decord av scikit-learn

In [ ]:
# 경로 설정 + Drive 마운트 (Colab)
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/BabyMon/dataset')   # ← 업로드 위치에 맞게 수정
assert BASE.exists(), f'{BASE} 없음 — Drive 업로드 위치를 확인하세요'

# 클립 폴더 자동 탐색 (clips/ 또는 resized_2/ 또는 BASE 바로 아래)
CLIPS = None
for cand in (BASE/'clips', BASE/'resized_2', BASE):
    if cand.is_dir() and next(cand.glob('*.mp4'), None):
        CLIPS = cand; break
assert CLIPS, f'{BASE} 아래에서 mp4 폴더를 못 찾음 (clips/ 또는 resized_2/)'
assert (BASE/'manifest.csv').exists(), \
    f'{BASE}/manifest.csv 없음 — 로컬 D:\\carved\\dataset\\manifest.csv 와 labels.csv 를 이 폴더로 업로드하세요'
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
print('clips:', CLIPS, '/', len(list(CLIPS.glob("*.mp4"))), '개')

In [ ]:
# 데이터: 균일 T프레임 샘플 (8초 클립이면 8프레임 = 1fps), 라벨은 02와 동일
import torch, numpy as np, pandas as pd, decord
LABEL_COLS = ["D1_nose_covered","D2_moving_freq","D3_eyes_open","D4_hands_out",
              "D5_pre_cry","D6_spit_up","D7_crying","D8_baby_sound","D9_mouthing",
              "C1_adult_hand","C2_baby_absent","C3_other_sound"]
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SIZE, EPOCHS = 224, 8

man = pd.read_csv(BASE/'manifest.csv')
lab = pd.read_csv(BASE/'labels.csv', dtype=str).drop_duplicates('file', keep='last')
df = man.merge(lab, on='file', how='left')
Y = pd.DataFrame(index=df.index, columns=LABEL_COLS, dtype=float)
for c in LABEL_COLS: Y[c] = pd.to_numeric(df.get(c), errors='coerce')
Y.loc[Y['D2_moving_freq'].isna(), 'D2_moving_freq'] = pd.to_numeric(df['weak_D2_moving'], errors='coerce')
Y.loc[Y['D8_baby_sound'].isna(), 'D8_baby_sound'] = pd.to_numeric(df['weak_D8_sound'], errors='coerce')
keep = Y.notna().any(axis=1)
df, Y = df[keep].reset_index(drop=True), Y[keep].reset_index(drop=True)
print('표본', len(df))

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def load_frames(path, t, size=SIZE):
    vr = decord.VideoReader(str(path))
    ix = np.linspace(0, len(vr)-1, t).astype(int)
    fr = torch.from_numpy(vr.get_batch(ix).asnumpy()).float()/255.
    fr = fr.permute(0,3,1,2)                                   # T,3,H,W
    fr = torch.nn.functional.interpolate(fr, size=(size, size), mode='bilinear', align_corners=False)
    return (fr - MEAN) / STD

class DS(torch.utils.data.Dataset):
    def __init__(self, sel, t):
        self.ix, self.t = np.where(sel)[0], t
    def __len__(self): return len(self.ix)
    def __getitem__(self, k):
        i = self.ix[k]
        return load_frames(CLIPS/df['file'].iloc[i], self.t), torch.tensor(Y.values[i], dtype=torch.float)

def masked_bce(logit, y):
    m = ~torch.isnan(y)
    return torch.nn.functional.binary_cross_entropy_with_logits(logit[m], y[m]) if m.any() else logit.sum()*0

In [ ]:
# 모델 동물원
import torchvision
from transformers import TimesformerModel, VideoMAEModel, CLIPVisionModel
N_OUT = len(LABEL_COLS)

class HFWrap(torch.nn.Module):
    """(B,T,3,H,W) → HF 비디오 백본 → CLS → 헤드"""
    def __init__(self, backbone, dim):
        super().__init__()
        self.b = backbone
        self.head = torch.nn.Linear(dim, N_OUT)
    def forward(self, v):
        h = self.b(pixel_values=v).last_hidden_state[:, 0]
        return self.head(h)

class CLIPFrozen(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch16')
        for p in self.clip.parameters(): p.requires_grad = False
        enc = torch.nn.TransformerEncoderLayer(768, 8, 1536, batch_first=True)
        self.temporal = torch.nn.TransformerEncoder(enc, 2)
        self.head = torch.nn.Linear(768, N_OUT)
    def forward(self, v):                                      # B,T,3,H,W
        b, t = v.shape[:2]
        with torch.no_grad():
            f = self.clip(pixel_values=v.flatten(0,1)).pooler_output   # B*T,768
        f = self.temporal(f.view(b, t, -1))
        return self.head(f.mean(1))

def r2p1d():
    m = torchvision.models.video.r2plus1d_18(weights='KINETICS400_V1')
    m.fc = torch.nn.Linear(m.fc.in_features, N_OUT)
    class W(torch.nn.Module):
        def __init__(s): super().__init__(); s.m = m
        def forward(s, v): return s.m(v.permute(0,2,1,3,4))    # B,3,T,H,W
    return W()

ZOO = {
    'r2plus1d':    dict(t=8,  bs=8, lr=3e-4, make=r2p1d),
    'timesformer': dict(t=8,  bs=4, lr=1e-4, make=lambda: HFWrap(
        TimesformerModel.from_pretrained('facebook/timesformer-base-finetuned-k400'), 768)),
    'videomae':    dict(t=16, bs=4, lr=1e-4, make=lambda: HFWrap(
        VideoMAEModel.from_pretrained('MCG-NJU/videomae-base-finetuned-kinetics'), 768)),
    'clip_frozen': dict(t=8,  bs=8, lr=3e-4, make=CLIPFrozen),
}

In [ ]:
# 공통 학습·평가 루프 — ZOO 순회
from sklearn.metrics import average_precision_score
results = {}
for name, cfg in ZOO.items():
    print(f'===== {name} (T={cfg["t"]}) =====')
    dl_tr = torch.utils.data.DataLoader(DS((df.split=="train").values, cfg['t']),
                                        cfg['bs'], shuffle=True, num_workers=2)
    dl_va = torch.utils.data.DataLoader(DS((df.split=="val").values, cfg['t']),
                                        cfg['bs'], shuffle=False, num_workers=2)
    model = cfg['make']().to(DEV)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg['lr'])
    scaler = torch.cuda.amp.GradScaler()
    best = 0
    for ep in range(EPOCHS):
        model.train()
        for v, y in dl_tr:
            v, y = v.to(DEV), y.to(DEV)
            with torch.cuda.amp.autocast():
                loss = masked_bce(model(v), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        model.eval(); P, T = [], []
        with torch.no_grad():
            for v, y in dl_va:
                with torch.cuda.amp.autocast():
                    P.append(torch.sigmoid(model(v.to(DEV))).float().cpu())
                T.append(y)
        P, T = torch.cat(P).numpy(), torch.cat(T).numpy()
        aps = {c: average_precision_score(T[m,i], P[m,i])
               for i, c in enumerate(LABEL_COLS)
               if (m := ~np.isnan(T[:,i])).sum() and len(set(T[m,i])) > 1}
        mAP = float(np.mean(list(aps.values()))) if aps else 0
        print(f'  ep{ep+1}: mAP={mAP:.3f}')
        if mAP > best:
            best = mAP; results[name] = {'mAP': mAP, **{k: round(v,3) for k,v in aps.items()}}
            torch.save(model.state_dict(), BASE/f'zoo_{name}.pt')
    del model; torch.cuda.empty_cache()

import json as _json
pd.DataFrame(results).T.to_csv(BASE/'zoo_results.csv')
print(_json.dumps(results, indent=1, ensure_ascii=False))

### 해석 가이드
- `zoo_results.csv` 에 모델×라벨 AP 저장 — **라벨별로 승자가 다를 수 있음** (움직임 계열은 CNN/TimeSformer, 정적 상태는 CLIP 계열이 유리한 경향).
- 1fps 샘플은 빠른 주기 동작(뻐끔 등)을 놓칠 수 있음 — 그런 라벨은 모듈 트랙(M3 시계열)과 비교할 것.
- 데이터가 적을수록 `clip_frozen` 이 안정적. 파인튜닝 계열이 과적합하면 EPOCHS 축소·백본 동결 후 헤드만 학습으로 전환.